# LLM API Tutorial (OpenAI, Anthropic, Gemini)

This tutorial demonstrates the **common structure** for using modern LLM APIs.

We will progressively build:

1. Basic API call
2. Add chat history
3. Add native web search


## 0. Setup Environment

## 1. Install Dependencies

# 2. Environment Variables

Create a `.env` file:


```env
OPENAI_API_KEY=your_openai_key
ANTHROPIC_API_KEY=your_anthropic_key
GEMINI_API_KEY=your_google_key
```

## 3. Common API Pattern

Use this mental model for every provider:

```text
client
   handles connection

model
   chooses capability

system prompt
   defines behavior

tools
   gives abilities

config
   tunes generation

user question
   defines task
```


Recommended structure:

```python
# 1. Client
client = ProviderClient()

# 2. Model
MODEL = "..."

# 3. Default prompt
SYSTEM_PROMPT = "You are a helpful assistant."

# 4. Optional tools
TOOLS = [...]

# 5. Optional generation settings
CONFIG = {...}

# 6. Dynamic user question
USER_QUESTION = "..."

# 7. API request
response = ...
```

## Part 1 — OpenAI

Official docs: https://platform.openai.com/docs

### 1.1 Basic API Call

In [12]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = OpenAI()

# -------------------
# 2. Model
# -------------------
MODEL = "gpt-4o-mini"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Tools
# -------------------
TOOLS = []

# -------------------
# 5. Config
# -------------------
CONFIG = {
    "temperature": 0.7
}

# -------------------
# 6. User Question
# -------------------
USER_QUESTION = "Explain what an LLM is."

# -------------------
# 7. Request
# -------------------
response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=TOOLS,
    input=USER_QUESTION,
    **CONFIG
)

print(response.output_text)

An LLM, or Large Language Model, is a type of artificial intelligence designed to understand and generate human language. These models are based on deep learning techniques, particularly using neural networks, and are trained on vast amounts of text data from books, articles, websites, and other sources.

### Key Features of LLMs:

1. **Scale**: LLMs are characterized by their large number of parameters, often in the billions or even trillions, which allows them to capture complex patterns and nuances in language.

2. **Training**: They are trained using unsupervised or semi-supervised learning methods, where the model learns to predict the next word in a sentence given the previous words. This process helps the model gain a broad understanding of language structure and context.

3. **Contextual Understanding**: LLMs can understand context, which allows them to generate coherent and contextually relevant responses. They can maintain context over longer conversations, making them useful


## 1.2 Add Chat History

OpenAI is **stateless**, so we store history manually.

In [13]:
client = OpenAI()

MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = "You are a helpful assistant."

history = []

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=history
    )

    answer = response.output_text
    print("AI:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })


User: My name is Thomas. What is your llm model
AI: Hello, Thomas! I'm based on OpenAI's GPT-3 model. How can I assist you today?
User: when is your cutoff date
AI: My knowledge cutoff date is September 2021. This means I don't have information on events or developments that occurred after that date. How can I help you with the information I do have?
User: what is my name
AI: Your name is Thomas. How can I assist you today, Thomas?
Exiting...


In [14]:
print("Chat history:")
for message in history:
    print(f"{message['role'].capitalize()}: {message['content']}")
    

Chat history:
User: My name is Thomas. What is your llm model
Assistant: Hello, Thomas! I'm based on OpenAI's GPT-3 model. How can I assist you today?
User: when is your cutoff date
Assistant: My knowledge cutoff date is September 2021. This means I don't have information on events or developments that occurred after that date. How can I help you with the information I do have?
User: what is my name
Assistant: Your name is Thomas. How can I assist you today, Thomas?


## 1.3 Add Native Web Search

In [15]:
client = OpenAI()

MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = "You are a helpful assistant."

TOOLS = [
    {
        "type": "web_search"
    }
]

USER_QUESTION = "What are the latest AI news today?"

response = client.responses.create(
    model=MODEL,
    instructions=SYSTEM_PROMPT,
    tools=TOOLS,
    input=USER_QUESTION
)

print(response.output_text)



Here are some of the latest developments in artificial intelligence as of May 12, 2026:

**U.S. Government Enhances AI Safety Measures**

The U.S. government is intensifying its oversight of advanced AI technologies by collaborating with major tech firms like Google DeepMind, Microsoft, and xAI. The Commerce Department's Center for Advanced Intelligence and Security Initiatives (CAISI) is conducting pre-deployment evaluations to assess the safety and national security implications of AI systems before they enter the market. This initiative aims to tighten cybersecurity protocols and may require pre-clearance for AI model deployment. ([axios.com](https://www.axios.com/2026/05/05/us-frontier-ai-testing-white-house-pivots-safety?utm_source=openai))

**Google Disrupts AI-Driven Cyberattack**

Google reported disrupting a criminal group's attempt to exploit an unknown digital vulnerability using artificial intelligence. This incident highlights the growing concern over AI's role in enhancin

## 1.4 Full OpenAI Chat with History and Search

In [18]:
client = OpenAI()

MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]

history = []

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history
    )

    answer = response.output_text
    print("AI:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })

User: my name is Thomas and my favorite food is curry chicken
AI: Nice to meet you, Thomas! Curry chicken is a delicious choice. Do you have a favorite recipe or type of curry chicken you enjoy?
User: what is your model trained on?
AI: I’m trained on a diverse range of text sources, including books, articles, and websites. This training helps me understand and generate human-like text and respond to various questions. My knowledge is current up until September 2021, and I aim to assist with a wide array of topics! If you have specific questions, feel free to ask!
User: which version of gpt?
AI: I’m based on OpenAI's GPT-4 architecture. If you have any questions about my capabilities or how I can assist you, just let me know!
User: what is yoru cutoff date?
AI: My knowledge cutoff date is September 2021. After that, I don’t have access to information or events that have occurred. If you have questions about anything up to that date, feel free to ask!
User: what is singapore q1 2026 gdp?

In [19]:
print("Chat history:")
for message in history:
    print(f"{message['role'].capitalize()}: {message['content']}")

Chat history:
User: my name is Thomas and my favorite food is curry chicken
Assistant: Nice to meet you, Thomas! Curry chicken is a delicious choice. Do you have a favorite recipe or type of curry chicken you enjoy?
User: what is your model trained on?
Assistant: I’m trained on a diverse range of text sources, including books, articles, and websites. This training helps me understand and generate human-like text and respond to various questions. My knowledge is current up until September 2021, and I aim to assist with a wide array of topics! If you have specific questions, feel free to ask!
User: which version of gpt?
Assistant: I’m based on OpenAI's GPT-4 architecture. If you have any questions about my capabilities or how I can assist you, just let me know!
User: what is yoru cutoff date?
Assistant: My knowledge cutoff date is September 2021. After that, I don’t have access to information or events that have occurred. If you have questions about anything up to that date, feel free to

## 1.5 Full OpenAI Chat with History and Search with Full Config

#### `temperature`

| Value  | Behavior                |
| ------ | ----------------------- |
| `0.0`  | deterministic           |
| `0.2`  | focused                 |
| `0.7`  | balanced (good default) |
| `1.0`  | creative                |
| `>1.2` | more random             |

#### `max_output_token`

| Task            | Suggested   |
| --------------- | ----------- |
| Short answers   | `200`       |
| Explanation     | `500`       |
| Long tutorial   | `1000–2000` |
| Code generation | `1500+`     |


In [23]:
client = OpenAI()

MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]
CONFIG = {
    "temperature": 0.2,
    "max_output_tokens": 1000,
}

history = []

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history,
        **CONFIG
    )

    answer = response.output_text
    print("AI:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })

User: My name is Thomas, what is your name
AI: I'm called Assistant! How can I help you today, Thomas?
User: What is yoru model based on?
AI: I'm based on a model developed by OpenAI called GPT (Generative Pre-trained Transformer). It's designed to understand and generate human-like text based on the input it receives. If you have any specific questions about how it works or its capabilities, feel free to ask!
User: which version
AI: I'm based on the GPT-4 architecture. If you have any questions about its features or capabilities, let me know!
User: who win the supoer bowl 2026
AI: 
## NFL Schedule
- Seattle Seahawks (SEA) @ New England Patriots (NE) on Sunday, Feb 08, 2026 at 03:30 PM PST. Final score: SEA 29 - NE 13


The Seattle Seahawks secured their second Super Bowl title by defeating the New England Patriots 29-13 in Super Bowl LX on February 8, 2026, at Levi's Stadium in Santa Clara, California. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Super_Bowl_LX?utm_source=openai))

In [24]:
print("Chat history:")
for message in history:
    print(f"{message['role'].capitalize()}: {message['content']}")

Chat history:
User: My name is Thomas, what is your name
Assistant: I'm called Assistant! How can I help you today, Thomas?
User: What is yoru model based on?
Assistant: I'm based on a model developed by OpenAI called GPT (Generative Pre-trained Transformer). It's designed to understand and generate human-like text based on the input it receives. If you have any specific questions about how it works or its capabilities, feel free to ask!
User: which version
Assistant: I'm based on the GPT-4 architecture. If you have any questions about its features or capabilities, let me know!
User: who win the supoer bowl 2026
Assistant: 
## NFL Schedule
- Seattle Seahawks (SEA) @ New England Patriots (NE) on Sunday, Feb 08, 2026 at 03:30 PM PST. Final score: SEA 29 - NE 13


The Seattle Seahawks secured their second Super Bowl title by defeating the New England Patriots 29-13 in Super Bowl LX on February 8, 2026, at Levi's Stadium in Santa Clara, California. ([en.wikipedia.org](https://en.wikipedia.

### ChatGPT API Changes for GPT-5

The biggest difference between **OpenAI “model 4” APIs (`gpt-4o`, `gpt-4o-mini`)** and **“model 5” APIs (`gpt-5`)** is not just model quality—it’s the **control style**.

Think of it like:

```text
GPT-4 family → "Generate text"
GPT-5 family → "Reason and generate"
```

---

### 1. Parameter differences

#### GPT-4 (`gpt-4o`, `gpt-4o-mini`)

Uses classic generation controls:

```python
response = client.responses.create(
    model="gpt-4o-mini",
    input="Explain transformers",
    temperature=0.7,
    max_output_tokens=500
)
```

Main knobs:

* `temperature` → creativity/randomness
* `top_p` → token sampling
* `max_output_tokens`

Mental model:

```text
How creative should the model be?
```

---

#### GPT-5 (`gpt-5`)

Uses **reasoning controls** instead:

```python
response = client.responses.create(
    model="gpt-5",
    input="Explain transformers",
    reasoning_effort="medium",
    verbosity="medium",
    max_output_tokens=500
)
```

Main knobs:

* `reasoning_effort`

  * `"low"`
  * `"medium"`
  * `"high"`

* `verbosity`

  * `"low"`
  * `"medium"`
  * `"high"`

Mental model:

```text
How much should the model think?
How detailed should the answer be?
```

---

### 2. Philosophy shift

#### GPT-4 API

You influence output **indirectly**.

Example:

```python
temperature=0.2
```

Means:

```text
Be more deterministic.
```

But it doesn’t explicitly control reasoning depth.

---

#### GPT-5 API

You influence output **explicitly**.

Example:

```python
reasoning_effort="high"
```

Means:

```text
Spend more compute thinking.
```

That’s much more intuitive.

---

### 3. Prompting style

#### GPT-4

Prompt engineering matters more.

You often write:

```text
Think step by step.
Be careful.
Double check your work.
```

To force better reasoning.

---

#### GPT-5

Less prompt hacking needed.

You can simply do:

```python
reasoning_effort="high"
```

The model internally allocates more reasoning.

---

### 4. Output behavior

#### GPT-4

Can be more:

* chatty
* variable
* creative

Good for:

* conversation
* brainstorming
* general assistants

---

#### GPT-5

Usually more:

* structured
* deliberate
* reliable
* better reasoning consistency

Good for:

* coding
* planning
* agent workflows
* tool use
* analytical tasks

---

### 5. Cost / latency behavior

#### GPT-4

Latency mostly depends on output length.

---

#### GPT-5

Latency depends on:

```text
reasoning_effort + output length
```

Example:

```python
reasoning_effort="high"
```

may take longer and cost more.

---

### Side-by-side comparison

| Feature             | GPT-4 (`4o`, `4o-mini`) | GPT-5                 |
| ------------------- | ----------------------- | --------------------- |
| Main control        | `temperature`           | `reasoning_effort`    |
| Detail control      | prompt-based            | `verbosity`           |
| Reasoning control   | indirect                | explicit              |
| Prompt engineering  | more important          | less important        |
| Creativity tuning   | yes                     | less emphasized       |
| Cost predictability | simpler                 | depends on reasoning  |
| Best for            | general chat            | reasoning-heavy tasks |

---

### Practical example

#### GPT-4 mindset

```python
response = client.responses.create(
    model="gpt-4o-mini",
    input="Solve this math problem",
    temperature=0.1
)
```

Meaning:

```text
Try to be consistent.
```

---

#### GPT-5 mindset

```python
response = client.responses.create(
    model="gpt-5",
    input="Solve this math problem",
    reasoning_effort="high"
)
```

Meaning:

```text
Think harder.
```

That’s a major conceptual upgrade.

---

### Which should you learn first?

For your tutorial, I’d suggest:

##### Start with **GPT-4o-mini**

Why:

* cheaper
* simpler
* most tutorials still use this style
* teaches classic LLM concepts (`temperature`)

---

##### Then learn **GPT-5**

To understand:

* reasoning models
* effort control
* modern OpenAI direction

---

### Easy analogy

```text
GPT-4:
"Answer this, and be less random."

GPT-5:
"Answer this, and think harder first."
```

That’s the core difference.


In [29]:
client = OpenAI()

MODEL = "gpt-5-nano"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]
CONFIG = {
    "reasoning_effort": "medium",
    "verbosity": "concise",
}

history = []

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history,
        **CONFIG
    )

    answer = response.output_text
    print("AI:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })

User: Hi


TypeError: Responses.create() got an unexpected keyword argument 'reasoning_effort'


# Part 2 — Anthropic Claude

Official docs: https://docs.anthropic.com

---

## 2.1 Basic API Call

```python
import anthropic
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = anthropic.Anthropic()

# -------------------
# 2. Model
# -------------------
MODEL = "claude-sonnet-4"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Tools
# -------------------
TOOLS = []

# -------------------
# 5. Config
# -------------------
CONFIG = {
    "max_tokens": 500,
    "temperature": 0.7
}

# -------------------
# 6. User Question
# -------------------
USER_QUESTION = "Explain transformers simply."

# -------------------
# 7. Request
# -------------------
response = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": USER_QUESTION
        }
    ],
    **CONFIG
)

print(response.content[0].text)
```

---

## 2.2 Add Chat History

Anthropic is also **stateless**.

```python
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()

MODEL = "claude-sonnet-4"
SYSTEM_PROMPT = "You are a helpful assistant."

history = []

while True:
    USER_QUESTION = input("You: ")

    history.append({
        "role": "user",
        "content": USER_QUESTION
    })

    response = client.messages.create(
        model=MODEL,
        system=SYSTEM_PROMPT,
        max_tokens=500,
        messages=history
    )

    answer = response.content[0].text
    print("Claude:", answer)

    history.append({
        "role": "assistant",
        "content": answer
    })
```

---

## 2.3 Add Native Web Search

```python
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()

MODEL = "claude-sonnet-4"

SYSTEM_PROMPT = "You are a helpful assistant."

TOOLS = [
    {
        "name": "web_search"
    }
]

USER_QUESTION = "Find recent developments in multimodal AI."

response = client.messages.create(
    model=MODEL,
    system=SYSTEM_PROMPT,
    tools=TOOLS,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": USER_QUESTION
        }
    ]
)

print(response.content[0].text)
```

---

# Part 3 — Google Gemini

Official docs: https://ai.google.dev/gemini-api/docs

---

## 3.1 Basic API Call

```python
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

# -------------------
# 1. Client
# -------------------
client = genai.Client()

# -------------------
# 2. Model
# -------------------
MODEL = "gemini-2.5-flash"

# -------------------
# 3. System Prompt
# -------------------
SYSTEM_PROMPT = "You are a helpful assistant."

# -------------------
# 4. Config
# -------------------
CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.7
)

# -------------------
# 5. User Question
# -------------------
USER_QUESTION = "Explain embeddings simply."

# -------------------
# 6. Request
# -------------------
response = client.models.generate_content(
    model=MODEL,
    contents=USER_QUESTION,
    config=CONFIG
)

print(response.text)
```

---

## 3.2 Add Chat History

Gemini supports **native chat sessions**.

```python
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

client = genai.Client()

MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = "You are a helpful assistant."

chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT
    )
)

while True:
    USER_QUESTION = input("You: ")

    response = chat.send_message(USER_QUESTION)

    print("Gemini:", response.text)
```

---

## 3.3 Add Native Google Search

```python
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

client = genai.Client()

MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = "You are a helpful assistant."

SEARCH_TOOL = types.Tool(
    google_search=types.GoogleSearch()
)

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    tools=[SEARCH_TOOL]
)

USER_QUESTION = "What happened in AI this week?"

response = client.models.generate_content(
    model=MODEL,
    contents=USER_QUESTION,
    config=CONFIG
)

print(response.text)
```



## Side-by-Side Comparison

| Concept | OpenAI | Anthropic | Gemini |
|---|---|---|---|
| System Prompt | `instructions=` | `system=` | `system_instruction=` |
| Model | `model=` | `model=` | `model=` |
| Tools | `tools=` | `tools=` | `tools=` |
| Config | `dict` | `dict` | `GenerateContentConfig()` |
| User Input | `input=` | `messages=` | `contents=` |
| Chat History | Manual | Manual | Native Chat |
| Web Search | `web_search` | `web_search` | `GoogleSearch()` |




# Key Takeaways

Always structure your LLM code like this:

```python
client
MODEL
SYSTEM_PROMPT
TOOLS
CONFIG
USER_QUESTION
response
```

This keeps your code:

- readable
- reusable
- provider-independent
- easy to upgrade

The API syntax changes, but the architecture stays the same.